# Task 4: Object Recognition in Video using SIFT

**Lab 06 solution**  
Run cells from top to bottom. Change only the paths in the configuration cell when using your own media.

In [1]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def show(img, title='', cmap=None, size=(12,7)):
    plt.figure(figsize=size)
    if img.ndim == 3:
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title); plt.axis('off'); plt.show()

## Objective
Locate a reference object in each video frame despite scale, rotation, and partial occlusion. The notebook writes an annotated output video instead of relying on GUI windows.

In [4]:
# ============================================================
# Initialize Reference Object + Video Tracking
# ============================================================

REFERENCE_PATH = 'object.png'
VIDEO_PATH = 'object_video.mp4'
OUTPUT_PATH = 'task4_recognized.mp4'


# ============================================================
# Load Reference Image and Video
# ============================================================

ref = cv.imread(
    REFERENCE_PATH
)

cap = cv.VideoCapture(
    VIDEO_PATH
)

if ref is None:
    raise FileNotFoundError(
        f"Reference image not found: {REFERENCE_PATH}"
    )

if not cap.isOpened():
    raise FileNotFoundError(
        f"Video could not be opened: {VIDEO_PATH}"
    )


# ============================================================
# Initialize SIFT
# ============================================================

sift = cv.SIFT_create(
    nfeatures=1800
)

gref = cv.cvtColor(
    ref,
    cv.COLOR_BGR2GRAY
)

kp1, des1 = sift.detectAndCompute(
    gref,
    None
)

if des1 is None or len(kp1) < 4:
    raise RuntimeError(
        "Not enough SIFT features in the reference object."
    )


print(
    "Reference keypoints:",
    len(kp1)
)


# ============================================================
# Read Video Properties
# ============================================================

fps = cap.get(
    cv.CAP_PROP_FPS
)

if fps <= 0:
    fps = 25


W = int(
    cap.get(
        cv.CAP_PROP_FRAME_WIDTH
    )
)

H = int(
    cap.get(
        cv.CAP_PROP_FRAME_HEIGHT
    )
)


print(
    "Video resolution:",
    f"{W} x {H}"
)

print(
    "Video FPS:",
    round(fps, 2)
)


# ============================================================
# Initialize Video Writer
# ============================================================

fourcc = cv.VideoWriter_fourcc(
    *'mp4v'
)

out = cv.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (W, H)
)

if not out.isOpened():
    raise RuntimeError(
        f"Could not create output video: {OUTPUT_PATH}"
    )


# ============================================================
# Initialize FLANN Matcher
# ============================================================

flann = cv.FlannBasedMatcher(
    dict(
        algorithm=1,
        trees=5
    ),
    dict(
        checks=60
    )
)


# ============================================================
# Tracking Counters
# ============================================================

detected_frames = 0
frame_count = 0

print("Video tracking initialized successfully.")


Reference keypoints: 94
Video resolution: 640 x 480
Video FPS: 25.0
Video tracking initialized successfully.


In [ ]:
# ============================================================
# Frame-by-Frame Object Recognition
# ============================================================

MIN_GOOD_MATCHES = 10
MIN_INLIERS = 8
MIN_INLIER_RATIO = 0.35


# ============================================================
# Process Video Frames
# ============================================================

try:

    while True:

        # ----------------------------------------------------
        # Read next frame
        # ----------------------------------------------------

        ok, frame = cap.read()

        if not ok:
            break

        frame_count += 1


        # ----------------------------------------------------
        # Detect SIFT features in current frame
        # ----------------------------------------------------

        gray_frame = cv.cvtColor(
            frame,
            cv.COLOR_BGR2GRAY
        )

        kp2, des2 = sift.detectAndCompute(
            gray_frame,
            None
        )


        # ----------------------------------------------------
        # Match reference descriptors to frame descriptors
        # ----------------------------------------------------

        if des2 is not None and len(kp2) >= 4:

            pairs = flann.knnMatch(
                des1,
                des2,
                k=2
            )

            # Lowe's ratio test
            good = []

            for pair in pairs:

                if len(pair) < 2:
                    continue

                m, n = pair

                if m.distance < 0.72 * n.distance:
                    good.append(m)


            # ------------------------------------------------
            # Estimate object location if enough matches exist
            # ------------------------------------------------

            if len(good) >= MIN_GOOD_MATCHES:

                src = np.float32([
                    kp1[m.queryIdx].pt
                    for m in good
                ]).reshape(-1, 1, 2)

                dst = np.float32([
                    kp2[m.trainIdx].pt
                    for m in good
                ]).reshape(-1, 1, 2)


                # ------------------------------------------------
                # Estimate homography with RANSAC
                # ------------------------------------------------

                M, inliers = cv.findHomography(
                    src,
                    dst,
                    cv.RANSAC,
                    5.0
                )


                if M is not None and inliers is not None:

                    inlier_mask = (
                        inliers.ravel().astype(bool)
                    )

                    inlier_count = int(
                        np.sum(inlier_mask)
                    )

                    inlier_ratio = (
                        inlier_count
                        / float(len(good))
                    )


                    # --------------------------------------------
                    # Validate homography
                    # --------------------------------------------

                    if (
                        inlier_count >= MIN_INLIERS
                        and inlier_ratio >= MIN_INLIER_RATIO
                    ):

                        # ----------------------------------------
                        # Reference object corners
                        # ----------------------------------------

                        h, w = ref.shape[:2]

                        box = np.float32([
                            [0, 0],
                            [w - 1, 0],
                            [w - 1, h - 1],
                            [0, h - 1]
                        ]).reshape(-1, 1, 2)


                        # ----------------------------------------
                        # Project reference object into frame
                        # ----------------------------------------

                        poly = cv.perspectiveTransform(
                            box,
                            M
                        )

                        poly = np.round(
                            poly
                        ).astype(np.int32)


                        # ----------------------------------------
                        # Check projected polygon
                        # ----------------------------------------

                        polygon_area = abs(
                            cv.contourArea(poly)
                        )

                        frame_area = W * H

                        if (
                            polygon_area > 100
                            and polygon_area < 1.5 * frame_area
                        ):

                            # ------------------------------------
                            # Draw detected object boundary
                            # ------------------------------------

                            cv.polylines(
                                frame,
                                [poly],
                                True,
                                (0, 255, 0),
                                4,
                                cv.LINE_AA
                            )


                            # ------------------------------------
                            # Recognition label
                            # ------------------------------------

                            cv.putText(
                                frame,
                                "OBJECT RECOGNIZED",
                                (20, 40),
                                cv.FONT_HERSHEY_SIMPLEX,
                                1,
                                (0, 255, 0),
                                3
                            )


                            # ------------------------------------
                            # Display matching statistics
                            # ------------------------------------

                            cv.putText(
                                frame,
                                (
                                    f"Inliers: "
                                    f"{inlier_count}/{len(good)} "
                                    f"({inlier_ratio:.2f})"
                                ),
                                (20, 75),
                                cv.FONT_HERSHEY_SIMPLEX,
                                0.7,
                                (0, 255, 0),
                                2
                            )

                            detected_frames += 1


        # ----------------------------------------------------
        # Write processed frame
        # ----------------------------------------------------

        out.write(frame)


finally:

    # Always release resources, even if an error occurs
    cap.release()
    out.release()


# ============================================================
# Final Results
# ============================================================

print(
    f"Saved {OUTPUT_PATH}"
)

print(
    f"Recognized in "
    f"{detected_frames}/{frame_count} frames"
)

if frame_count > 0:

    detection_rate = (
        detected_frames
        / frame_count
        * 100
    )

    print(
        f"Detection rate: "
        f"{detection_rate:.1f}%"
    )


Saved task4_recognized.mp4
Recognized in 69/75 frames
Detection rate: 92.0%


## Notes
The ratio test handles descriptor ambiguity; RANSAC homography handles outliers and yields the bounding polygon. Increase the minimum inliers to reduce false alarms, or lower it slightly for severe occlusion.